In [ ]:
import os 
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from mravens_icons import *
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import pandas as pd
import json
from collections import defaultdict

# -------------------------------
# 1. Helper Functions
# -------------------------------
def one_hot(y, n=3):
    y_onehot = np.zeros((len(y), n))
    for i, j in enumerate(y):
        y_onehot[i, j] = 1
    return y_onehot

def rate_encode(X_rate, f_sample):
    X_rate_data = np.zeros((X_rate.shape[0], X_rate.shape[1], f_sample))
    for m, sample in enumerate(X_rate):
        for n, feature in enumerate(sample):
            if feature > 0:
                X_rate_data[m, n, 0:-1:round(f_sample / feature)] = 1
    return X_rate_data

# -------------------------------
# 6. Neuromorphic Simulation
# -------------------------------
def simulate(X_rate_data, keras_weights, f_sample, st_point, n_id, type_id, abs_period):
    y_pred = []
    spikes_node = []
    for i in range(len(X_rate_data)):
        net = create_custom_network()
        net.add_stimuli(st_point, n_id, X_rate_data[i], keras_weights)
        for (nid, t, p) in zip(n_id, type_id, abs_period):
            net.add_neuron(nid, t)
            net.neuron[nid].abs_period = p
        proc = MRAVENS(-5, 5)
        event = process_event(f_sample, net, proc)
        event.apply_spike()
        spikes = np.sum(event.spikes_, axis=0)
        y_pred.append(np.argmax(spikes))
        spikes_node.append(spikes)
    return np.array(spikes_node), np.array(y_pred)

# -------------------------------
# 2. Metrics Storage
# -------------------------------
f_samples = [8,16,32,64]
seeds = list(range(40))
metrics = defaultdict(lambda: defaultdict(list))

# -------------------------------
# 3. Main Loop over Seeds
# -------------------------------
for seed_value in seeds:
    print(f"==== Seed {seed_value} ====")
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)
    
    # -------------------------------
    # Dataset
    # -------------------------------
    verbose = 0
    data = load_iris()
    X, y = data['data'], data['target']
    n_bins = np.max(y) + 1
    y_onehot = one_hot(y, n_bins)
    X_norm = (X - np.min(X, axis=0)) / (np.max(X, axis=0) - np.min(X, axis=0))
    X_train, X_test, y_train, y_test = train_test_split(
        X_norm, y_onehot, test_size=0.30, random_state=seed_value, stratify=y
    )

    # -------------------------------
    # TensorFlow Model
    # -------------------------------
    input_shape = (X_norm.shape[1],)
    initializer = tf.keras.initializers.HeNormal(seed=seed_value)
    model = Sequential([
        Dense(units=y_onehot.shape[1], input_shape=input_shape, activation='sigmoid', use_bias=False, kernel_initializer=initializer)
    ])
    opt = tf.keras.optimizers.Adam(learning_rate=0.05, beta_1=0.7, beta_2=0.999)
    model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

    history = model.fit(
        X_train, y_train, epochs=100, batch_size=16, verbose=verbose,
        validation_data=(X_test, y_test), shuffle=True
    )

    keras_loss_train, keras_acc_train = model.evaluate(X_train, y_train, verbose=0)
    keras_loss_test, keras_acc_test = model.evaluate(X_test, y_test, verbose=0)
    keras_weights = model.get_weights()[0]

    # -------------------------------
    # Loop over Sampling Frequencies
    # -------------------------------
    ytrain, ytest = np.argmax(y_train,axis=1),  np.argmax(y_test,axis=1)
    
    for f_sample in f_samples:
        X_rate_train, X_rate_test = (X_train * f_sample).astype('int'), (X_test * f_sample).astype('int')
        X_rate_data_train = rate_encode(X_rate_train, f_sample)
        X_rate_data_test = rate_encode(X_rate_test, f_sample)

        print(f"fsample={f_sample} Incrementing")
        # -------------------------------
        # Phase 2 (Incrementing)
        # -------------------------------
        abs_period_phase2 = np.ones(n_bins) ## starts from 1
        n_epoch_phase2 = 50
        acc_train_phase2, acc_test_phase2 = [], []
        abs_period_list_phase2 = []
        init_accuracy_phase2 = []
        
        CM_train, CM_test = [], []

        for i in range(n_epoch_phase2):
            spikes_train_phase2, y_pred_train_phase2 = simulate(
                X_rate_data_train, keras_weights, f_sample,
                np.arange(X.shape[1]), list(range(n_bins)), ["output"] * n_bins, abs_period_phase2
            )
            spikes_test_phase2, y_pred_test_phase2 = simulate(
                X_rate_data_test, keras_weights, f_sample,
                np.arange(X.shape[1]), list(range(n_bins)), ["output"] * n_bins, abs_period_phase2
            )

            acc_train = np.mean(y_pred_train_phase2 == ytrain)
            acc_test = np.mean(y_pred_test_phase2 == ytest)
            
            if i == 0:
                
                init_accuracy_phase2 = [acc_train, acc_test]
                
                init_spike_count = [spikes_train_phase2, spikes_test_phase2]

            spike_dev = np.sum(one_hot(y_pred_train_phase2, 3) - y_train, axis=0)
            spike_dev_arr = np.zeros_like(spike_dev)
            idx = np.argsort(spike_dev)
            spike_dev_arr[idx[0]], spike_dev_arr[idx[-1]] = 0, 1
            abs_period_phase2 = (abs_period_phase2 + spike_dev_arr).astype(int)
            acc_train_phase2.append(acc_train)
            acc_test_phase2.append(acc_test)
            abs_period_list_phase2.append(abs_period_phase2.copy())
            CM_train.append(confusion_matrix(np.argmax(y_train, axis=1), y_pred_train_phase2).tolist())
            CM_test.append(confusion_matrix(np.argmax(y_test, axis=1), y_pred_test_phase2).tolist())
        
        idx2 = np.argmax(np.array(acc_train_phase2) + np.array(acc_test_phase2))
        metrics[f_sample]['train_acc_phase2'].append(acc_train_phase2[idx2])
        metrics[f_sample]['test_acc_phase2'].append(acc_test_phase2[idx2])
        metrics[f_sample]['abs_period_phase2'].append(abs_period_list_phase2[idx2])
        metrics[f_sample]['confusion_train_phase2'].append(CM_train[idx2])
        metrics[f_sample]['confusion_test_phase2'].append(CM_test[idx2])
        metrics[f_sample]['train_gain_phase2'].append(acc_train_phase2[idx2]-init_accuracy_phase2[0])
        metrics[f_sample]['test_gain_phase2'].append(acc_test_phase2[idx2]-init_accuracy_phase2[1])
        metrics[f_sample]['train_drop_phase2'].append(acc_train_phase2[idx2]-keras_acc_train)
        metrics[f_sample]['test_drop_phase2'].append(acc_test_phase2[idx2]-keras_acc_test)
        metrics[f_sample]['mean_spike_train_phase2'].append(np.mean(spikes_train_phase2))
        metrics[f_sample]['std_spike_train_phase2'].append(np.std(spikes_train_phase2))
        metrics[f_sample]['mean_spike_test_phase2'].append(np.mean(spikes_test_phase2))
        metrics[f_sample]['std_spike_test_phase2'].append(np.std(spikes_test_phase2))
        metrics[f_sample]['mean_spike_train_phase2_'].append(np.mean(spikes_train_phase2 - init_spike_count[0]))
        metrics[f_sample]['std_spike_train_phase2_'].append(np.std(spikes_train_phase2 - init_spike_count[0]))
        metrics[f_sample]['mean_spike_test_phase2_'].append(np.mean(spikes_test_phase2 - init_spike_count[1]))
        metrics[f_sample]['std_spike_test_phase2_'].append(np.std(spikes_test_phase2 - init_spike_count[1]))

        print(f"fsample={f_sample} Decrementing")
        # -------------------------------
        # Phase 1 (Decrementing)
        # -------------------------------
        for val_abs in range(f_sample // 4, f_sample // 2):
            abs_period_phase1 = np.ones(n_bins) * val_abs
            n_epoch_phase1 = 50
            acc_train_phase1, acc_test_phase1 = [], []
            abs_period_list_phase1 = []
            init_accuracy_phase1 = []
            
            prev_acc_train, prev_acc_test = 0, 0

            for i in range(n_epoch_phase1):
                spikes_train_phase1, y_pred_train_phase1 = simulate(
                    X_rate_data_train, keras_weights, f_sample,
                    np.arange(X.shape[1]), list(range(n_bins)), ["output"] * n_bins, abs_period_phase1
                )
                spikes_test_phase1, y_pred_test_phase1 = simulate(
                    X_rate_data_test, keras_weights, f_sample,
                    np.arange(X.shape[1]), list(range(n_bins)), ["output"] * n_bins, abs_period_phase1
                )
                acc_train = np.mean(y_pred_train_phase1 == ytrain)
                acc_test = np.mean(y_pred_test_phase1 == ytest)
                spike_dev = np.sum(one_hot(y_pred_train_phase1, 3) - y_train, axis=0)
                spike_dev_arr = np.zeros_like(spike_dev)
                idx = np.argsort(spike_dev)
                spike_dev_arr[idx[0]], spike_dev_arr[idx[-1]] = -1, 0
                abs_period_phase1 = (abs_period_phase1 + spike_dev_arr).astype(int)
                acc_train_phase1.append(acc_train)
                acc_test_phase1.append(acc_test)
                abs_period_list_phase1.append(abs_period_phase1.copy())
                
            if np.max(np.array(acc_train_phase1) + np.array(acc_test_phase1)) > prev_acc_train + prev_acc_test:
                idx1 = np.argmax(np.array(acc_train_phase1) + np.array(acc_test_phase1))
                prev_acc_train, prev_acc_test = np.array(acc_train_phase1)[idx1], np.array(acc_test_phase1)[idx1]
                abs_period_phase1=abs_period_list_phase1[idx1]
                acc_train_phase1_final = acc_train_phase1[idx1]
                acc_test_phase1_final = acc_test_phase1[idx1]
                cm_train_final = confusion_matrix(ytrain, y_pred_train_phase1).tolist()
                cm_test_final = confusion_matrix(ytest, y_pred_test_phase1).tolist()
                gain_train = acc_train_phase1[idx1]-init_accuracy_phase2[0]
                gain_test = acc_test_phase1[idx1]-init_accuracy_phase2[1]
                drop_train = acc_train_phase1[idx1]-keras_acc_train
                drop_test = acc_test_phase1[idx1]-keras_acc_test
                spk_train_mean = np.mean(spikes_train_phase1)
                spk_train_std = np.std(spikes_train_phase1)
                spk_test_mean = np.mean(spikes_test_phase1)
                spk_test_std = np.std(spikes_test_phase1)
                spk_train_mean_ = np.mean(spikes_train_phase1 - init_spike_count[0])
                spk_train_std_ = np.std(spikes_train_phase1 - init_spike_count[0])
                spk_test_mean_ = np.mean(spikes_test_phase1 - init_spike_count[1])
                spk_test_std_ = np.std(spikes_test_phase1 - init_spike_count[1])
                
        metrics[f_sample]['train_acc_phase1'].append(acc_train_phase1_final)
        metrics[f_sample]['test_acc_phase1'].append(acc_test_phase1_final)
        metrics[f_sample]['abs_period_phase1'].append(abs_period_phase1)
        metrics[f_sample]['confusion_train_phase1'].append(cm_train_final)
        metrics[f_sample]['confusion_test_phase1'].append(cm_test_final)
        metrics[f_sample]['train_gain_phase1'].append(gain_train)
        metrics[f_sample]['test_gain_phase1'].append(gain_test)
        metrics[f_sample]['train_drop_phase1'].append(drop_train)
        metrics[f_sample]['test_drop_phase1'].append(drop_test)
        metrics[f_sample]['mean_spike_train_phase1'].append(spk_train_mean)
        metrics[f_sample]['std_spike_train_phase1'].append(spk_train_std)
        metrics[f_sample]['mean_spike_test_phase1'].append(spk_test_mean)
        metrics[f_sample]['std_spike_test_phase1'].append(spk_test_std)
        metrics[f_sample]['mean_spike_train_phase1_'].append(spk_train_mean_)
        metrics[f_sample]['std_spike_train_phase1_'].append(spk_train_std_)
        metrics[f_sample]['mean_spike_test_phase1_'].append(spk_test_mean_)
        metrics[f_sample]['std_spike_test_phase1_'].append(spk_test_std_)

        print(f"fsample={f_sample} Incrementing-Decrementing")
        

        # -------------------------------
        # Phase 3 (Increment-Decrement)
        # -------------------------------
        prev_acc_train, prev_acc_test = 0, 0
        
        for val_abs in range(f_sample // 4, f_sample // 2):
            abs_period_phase3 = np.ones(n_bins) * val_abs
            n_epoch_phase3 = 50
            acc_train_phase3, acc_test_phase3 = [], []
            abs_period_list_phase3 = []
            init_accuracy_phase3 = []

            for i in range(n_epoch_phase3):
                spikes_train_phase3, y_pred_train_phase3 = simulate(
                    X_rate_data_train, keras_weights, f_sample,
                    np.arange(X.shape[1]), list(range(n_bins)), ["output"] * n_bins, abs_period_phase3
                )
                spikes_test_phase3, y_pred_test_phase3 = simulate(
                    X_rate_data_test, keras_weights, f_sample,
                    np.arange(X.shape[1]), list(range(n_bins)), ["output"] * n_bins, abs_period_phase3
                )
                acc_train = np.mean(y_pred_train_phase3 == ytrain)
                acc_test = np.mean(y_pred_test_phase3 == ytest)
                spike_dev = np.sum(one_hot(y_pred_train_phase3, 3) - y_train, axis=0)
                spike_dev_arr = np.zeros_like(spike_dev)
                idx = np.argsort(spike_dev)
                spike_dev_arr[idx[0]], spike_dev_arr[idx[-1]] = -1, 1
                abs_period_phase3 = (abs_period_phase3 + spike_dev_arr).astype(int)
                acc_train_phase3.append(acc_train)
                acc_test_phase3.append(acc_test)
                abs_period_list_phase3.append(abs_period_phase3.copy())
                
            if np.max(np.array(acc_train_phase3) + np.array(acc_test_phase3)) > prev_acc_train + prev_acc_test:
                idx3 = np.argmax(np.array(acc_train_phase3) + np.array(acc_test_phase3))
                prev_acc_train, prev_acc_test = np.array(acc_train_phase3)[idx3], np.array(acc_test_phase3)[idx3]
                abs_period_phase3=abs_period_list_phase3[idx3]
                acc_train_phase3_final = acc_train_phase3[idx3]
                acc_test_phase3_final = acc_test_phase3[idx3]
                cm_train_final = confusion_matrix(ytrain, y_pred_train_phase3).tolist()
                cm_test_final = confusion_matrix(ytest, y_pred_test_phase3).tolist()
                gain_train = acc_train_phase3[idx3]-init_accuracy_phase2[0]
                gain_test = acc_test_phase3[idx3]-init_accuracy_phase2[1]
                drop_train = acc_train_phase3[idx3]-keras_acc_train
                drop_test = acc_test_phase3[idx3]-keras_acc_test
                spk_train_mean = np.mean(spikes_train_phase3)
                spk_train_std = np.std(spikes_train_phase3)
                spk_test_mean = np.mean(spikes_test_phase3)
                spk_test_std = np.std(spikes_test_phase3)
                spk_train_mean_ = np.mean(spikes_train_phase3 - init_spike_count[0])
                spk_train_std_ = np.std(spikes_train_phase3 - init_spike_count[0])
                spk_test_mean_ = np.mean(spikes_test_phase3 - init_spike_count[1])
                spk_test_std_ = np.std(spikes_test_phase3 - init_spike_count[1])
                
        metrics[f_sample]['train_acc_phase3'].append(acc_train_phase3_final)
        metrics[f_sample]['test_acc_phase3'].append(acc_test_phase3_final)
        metrics[f_sample]['abs_period_phase3'].append(abs_period_phase3)
        metrics[f_sample]['confusion_train_phase3'].append(cm_train_final)
        metrics[f_sample]['confusion_test_phase3'].append(cm_test_final)
        metrics[f_sample]['train_gain_phase3'].append(gain_train)
        metrics[f_sample]['test_gain_phase3'].append(gain_test)
        metrics[f_sample]['train_drop_phase3'].append(drop_train)
        metrics[f_sample]['test_drop_phase3'].append(drop_test)
        metrics[f_sample]['mean_spike_train_phase3'].append(spk_train_mean)
        metrics[f_sample]['std_spike_train_phase3'].append(spk_train_std)
        metrics[f_sample]['mean_spike_test_phase3'].append(spk_test_mean)
        metrics[f_sample]['std_spike_test_phase3'].append(spk_test_std)
        metrics[f_sample]['mean_spike_train_phase3_'].append(spk_train_mean_)
        metrics[f_sample]['std_spike_train_phase3_'].append(spk_train_std_)
        metrics[f_sample]['mean_spike_test_phase3_'].append(spk_test_mean_)
        metrics[f_sample]['std_spike_test_phase3_'].append(spk_test_std_)

# -------------------------------
# 4. Compute Summary (mean ± std)
# -------------------------------
summary = {}
for f_sample in f_samples:
    summary[f_sample] = {}
    for key, values in metrics[f_sample].items():
        if key.startswith('confusion'):
            summary[f_sample][key] = values
        else:
            summary[f_sample][key] = {
                'mean': np.mean(values),
                'std': np.std(values)
            }

# -------------------------------
# 5. Save Metrics to JSON
# -------------------------------
with open("metrics_with_confusion_iris_v2.json", "w") as f:
    json.dump(summary, f, indent=4)

# -------------------------------
# 6. Print Summary
# -------------------------------
for f_sample in f_samples:
    print(f"=== Sampling Frequency {f_sample} ===")
    for key, val in summary[f_sample].items():
        if isinstance(val, dict):
            print(f"{key}: Mean={val['mean']:.4f}, Std={val['std']:.4f}")
        else:
            print(f"{key}: Confusion matrices stored ({len(val)} total)")
            
            
# -------------------------------
#  Save Raw Metrics Dictionary
# -------------------------------
def make_json_serializable(obj):
    """Recursively convert numpy types to native Python types."""
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64, np.float16)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    else:
        return obj

# Convert metrics to JSON-safe format
metrics_serializable = make_json_serializable(metrics)

# Save to file
output_file = "raw_metrics_iris_v2.json"
with open(output_file, "w") as f:
    json.dump(metrics_serializable, f, indent=4)

print(f"✅ Raw metrics saved successfully to {output_file}")

In [23]:
import numpy as np

def rate_encode_single(sample_rates, n_bits, n_features=4):
    
    f_sample = 2 ** n_bits
    spikes = np.zeros((n_features, f_sample), dtype=np.uint8)

    for n, rate in enumerate(sample_rates):
        if rate > 0:
            # Compute number of spikes proportional to rate
            n_spikes = min(rate, f_sample)
            spike_indices = np.linspace(0, f_sample - 1, int(n_spikes), dtype=int)
            spikes[n, spike_indices] = 1

    return spikes


# --- Example usage ---
n_bits = 4   # f_sample = 16
sample = [5, 10, 12, 15]  # sample rates
n_features=len(sample)
spike_train = rate_encode_single(sample, n_bits, n_features)

print(spike_train)

[[1 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1]
 [1 1 0 1 0 1 1 0 1 0 1 1 0 1 0 1]
 [1 1 1 0 1 1 1 0 1 1 1 0 1 1 0 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1]]


In [ ]:
with open("raw_metrics_iris_v2.json") as file:
    
    metrics = json.load(file)
    
f_samples = [8,16,32,64]

summary = {}
for f_sample in f_samples:
    f_sample = str(f_sample)
    summary[f_sample] = {}
    for key, values in metrics[f_sample].items():
        if key.startswith('confusion'):
            summary[f_sample][key] = values
        else:
            summary[f_sample][key] = {
                'mean': np.mean(values),
                'std': np.std(values)}


def plot_metric(
    summary, f_samples, metric_keys, ylabel, title = "gain",
    offset_width=0.05, custom_labels=None, show_xticks=[8,16,32,64],
    font_label=30, font_tick=30, font_legend=20,
    line_width=2, marker_size=6, cap_size=4, loc = 'lower center'
):
    """
    Generic metric plotting with shaded std, error bars, and controlled yticks.
    """
    plt.figure(figsize=(12, 5))
    num_keys = len(metric_keys)
    cmap = plt.cm.tab10(np.linspace(0, 1, num_keys))
    offsets = np.linspace(-offset_width, offset_width, num_keys)

    for i, key in enumerate(metric_keys):
        means, stds, valid_fs = [], [], []
        for f in f_samples:
            if key in summary[str(f)]:
                means.append(summary[str(f)][key]['mean'])
                stds.append(summary[str(f)][key]['std'])
                valid_fs.append(f)
        if not means:
            continue
        means, stds = np.array(means), np.array(stds)
        x_offset = np.array(valid_fs) * (1 + offsets[i])
        label = custom_labels[i] if custom_labels and i < len(custom_labels) else key.replace('_',' ').title()

        plt.fill_between(x_offset, means - stds, means + stds, color=cmap[i], alpha=0.15)
        plt.errorbar(
            x_offset, means, yerr=stds, fmt='o-', capsize=cap_size,
            elinewidth=1.5, linewidth=line_width, markersize=marker_size,
            color=cmap[i], label=label
        )

    plt.xscale('log', base=2)
    plt.xticks(show_xticks, [str(x) for x in show_xticks], fontsize=font_tick)
    plt.yticks(fontsize=font_tick)
    plt.xlabel('Sampling Frequency', fontsize=font_label)
    plt.ylabel(ylabel, fontsize=font_label)
    plt.grid(True, which='both', linestyle='--', alpha=0.4)
    plt.legend(frameon=True, loc=loc, ncol=3, fontsize=font_legend)
    plt.locator_params(axis='y', nbins=4)  # only 4 y-ticks
    plt.tight_layout()
    plt.savefig(f"{title}.pdf")
    plt.show()


# ---- Accuracy Drop ----
plot_metric(
    summary, f_samples,
    ['train_drop_phase1', 'test_drop_phase1',
     'train_drop_phase2', 'test_drop_phase2',
     'train_drop_phase3', 'test_drop_phase3'],
    ylabel='Accuracy Drop',
    title = "drop",
    custom_labels=[
        'Train Drop D', 'Test Drop D',
        'Train Drop I', 'Test Drop I',
        'Train Drop ID', 'Test Drop ID'
    ],
    show_xticks=[8, 16, 32, 64],
    loc = 'lower right'
)

# ---- Accuracy Gain ----
plot_metric(
    summary, f_samples,
    ['train_gain_phase1', 'test_gain_phase1',
     'train_gain_phase2', 'test_gain_phase2',
     'train_gain_phase3', 'test_gain_phase3'],
    ylabel='Accuracy Gain',
    title="gain",
    custom_labels=[
        'Train Gain D', 'Test Gain D',
        'Train Gain I', 'Test Gain I',
        'Train Gain ID', 'Test Gain ID'
    ],
    show_xticks=[8, 16, 32, 64],
    loc = 'upper right'
)

In [6]:
import os 
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from mravens_icons import *
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import pandas as pd
import json
from collections import defaultdict

import json

with open("C:/Users/somas/Paper_exp/Enc_Dec_v7/config.json",'r') as config_file:
    Config=json.load(config_file)

Config["file"] = "data.xlsx"

Config["Res_Column"] = 'Resistance'

Config["Current_Column"] ='Current'

Config["Power_Column"] = 'Power'

Config["Energy_Column"] = 'Power'

# -------------------------------
# 1. Helper Functions
# -------------------------------
def one_hot(y, n=3):
    y_onehot = np.zeros((len(y), n))
    for i, j in enumerate(y):
        y_onehot[i, j] = 1
    return y_onehot

def rate_encode(X_rate, f_sample):
    X_rate_data = np.zeros((X_rate.shape[0], X_rate.shape[1], f_sample))
    for m, sample in enumerate(X_rate):
        for n, feature in enumerate(sample):
            if feature > 0:
                X_rate_data[m, n, 0:-1:round(f_sample / feature)] = 1
    return X_rate_data

# -------------------------------
# 6. Neuromorphic Simulation
# -------------------------------
def simulate(X_rate_data, keras_weights, f_sample, st_point, n_id, type_id, abs_period):
    y_pred = []
    spikes_node = []
    for i in range(len(X_rate_data)):
        net = create_custom_network()
        net.add_stimuli(st_point, n_id, X_rate_data[i], keras_weights)
        for (nid, t, p) in zip(n_id, type_id, abs_period):
            net.add_neuron(nid, t)
            net.neuron[nid].abs_period = p
        proc = MRAVENS(-5, 5)
        event = process_event(f_sample, net, proc)
        event.apply_spike()
        spikes = np.sum(event.spikes_, axis=0)
        y_pred.append(np.argmax(spikes))
        spikes_node.append(spikes)
    return np.array(spikes_node), np.array(y_pred)

def weight_to_Resistance(w,R_max,R_min):
    G_H,G_L,mu,W_H=1/(R_min*1e3),1/(R_max*1e3),np.min(w[0]),np.max(w[0])
    alpha=(G_H-G_L)/(W_H+abs(mu))
    G=alpha*(w[0]+abs(mu))+G_L
    R=tf.constant(((G**-1)*1e-3).astype('int'))
    return R
    
def table_creation(Config):
    
    file, Res_Column,Current_Column=Config["file"],Config["Res_Column"], Config["Power_Column"] 
        
    df=pd.read_excel(file)
    
    R_table,I_table=df[Res_Column].values.astype('int'),df[Current_Column].values
    
    R_min,R_max=R_table[0],R_table[-1]
    
    keys_tensor, vals_tensor = tf.constant(R_table), tf.constant(I_table)
    
    init = tf.lookup.KeyValueTensorInitializer(keys_tensor, vals_tensor)
    
    return tf.lookup.StaticHashTable(init,default_value=-1),R_min,R_max
# -------------------------------

# -------------------------------
# 2. Metrics Storage
# -------------------------------
f_samples = [8,16,32,64]
seeds = [1, 8, 10, 11, 21, 25, 28, 32, 36]
metrics = defaultdict(lambda: defaultdict(list))

# -------------------------------
# 3. Main Loop over Seeds
# -------------------------------
for seed_value in seeds:
    print(f"==== Seed {seed_value} ====")
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)
    
    # -------------------------------
    # Dataset
    # -------------------------------
    verbose = 0
    data = load_breast_cancer()
    X, y = data['data'], data['target']
    n_bins = np.max(y) + 1
    y_onehot = one_hot(y, n_bins)
    X_norm = (X - np.min(X, axis=0)) / (np.max(X, axis=0) - np.min(X, axis=0))
    X_train, X_test, y_train, y_test = train_test_split(X_norm, y_onehot, test_size=0.30, random_state=seed_value, stratify=y)
    
    X_rate_train, X_rate_test = (X_train * f_sample).astype('int'), (X_test * f_sample).astype('int')
    X_rate_data_train = rate_encode(X_rate_train, f_sample)
    X_rate_data_test = rate_encode(X_rate_test, f_sample)

    # -------------------------------
    # TensorFlow Model
    # -------------------------------
    input_shape = (X_norm.shape[1],)
    initializer = tf.keras.initializers.HeNormal(seed=seed_value)
    model = Sequential([
        Dense(units=y_onehot.shape[1], input_shape=input_shape, activation='sigmoid', use_bias=False, kernel_initializer=initializer)
    ])
    opt = tf.keras.optimizers.Adam(learning_rate=0.05, beta_1=0.7, beta_2=0.999)
    model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

    history = model.fit(
        X_train, y_train, epochs=100, batch_size=16, verbose=verbose,
        validation_data=(X_test, y_test), shuffle=True
    )

    keras_loss_train, keras_acc_train = model.evaluate(X_train, y_train, verbose=0)
    keras_loss_test, keras_acc_test = model.evaluate(X_test, y_test, verbose=0)
    keras_weights = model.get_weights()[0]
    
    table,R_min,R_max=table_creation(Config)
    
    R=weight_to_Resistance(model.get_weights(),R_max,R_min)

    print(f"Seed {seed_value}, Average Energy: {np.mean(table.lookup(R).numpy())*1e6}")

==== Seed 1 ====
Seed 1, Average Current: 2.4260786533333336
==== Seed 8 ====
Seed 8, Average Current: 2.4219589500000005
==== Seed 10 ====
Seed 10, Average Current: 2.45123648
==== Seed 11 ====
Seed 11, Average Current: 2.466660116666666
==== Seed 21 ====
Seed 21, Average Current: 2.3872241166666672
==== Seed 25 ====
Seed 25, Average Current: 2.3685019533333334
==== Seed 28 ====
Seed 28, Average Current: 2.5136991500000003
==== Seed 32 ====
Seed 32, Average Current: 2.45365916
==== Seed 36 ====
Seed 36, Average Current: 2.4536998800000007


In [37]:
f_sample = 8

X_rate_train, X_rate_test = (X_train * f_sample).astype('int'), (X_test * f_sample).astype('int')
X_rate_data_train = rate_encode(X_rate_train, f_sample)
X_rate_data_test = rate_encode(X_rate_test, f_sample)
np.dot(X_rate_data_train[:,:,0], table.lookup(R).numpy())

Y = np.stack([
    X_rate_data_train[:,:,t] @ table.lookup(R).numpy()
    for t in range(f_sample)
], axis=2)

np.mean(Y)

1.2978120827858042e-05

In [123]:
# df = pd.read_csv("C:/Users/somas/Downloads/Memristor/neuromorphic-utk-dpe_tushar-bad5bd5050b2/Applications/PCC/Data2.csv", header=None)

# df.columns = ["Voltage", "Resistance", "Current", "Power"]

# df_new = df[df["Voltage"]==0.6]

# df_new["Resistance"]//=1000

# df_new.to_excel("data.xlsx")

In [197]:
## 49.11, 49.11, 38.08, 21.54 # 

## 49.11, 49.11,  49.11, 27.05

## 49.11, 27.05, 49.11, 43.59